Create clean tables that can be used for embeddings and derive relation. apply any transformation here

In [0]:
##read bronze data
from pyspark.sql.functions import col, explode, from_json, expr
from pyspark.sql.types import ArrayType, IntegerType, StructType, StructField, StringType

bronze_movies = spark.table("moviebuff.default.bronze_movies_raw")
bronze_genres = spark.table("moviebuff.default.bronze_genres")
bronze_providers = spark.table("moviebuff.default.bronze_watch_providers_raw")

In [0]:
spark.sql("USE moviebuff.default")

In [0]:
movie_schema = StructType([
    StructField("id", IntegerType()),
    StructField("title", StringType()),
    StructField("overview", StringType()),
    StructField("release_date", StringType()),
    StructField("vote_average", StringType()),
    StructField("vote_count", IntegerType()),
    StructField("popularity", StringType()),
    StructField("original_language", StringType()),
    StructField("genre_ids", ArrayType(IntegerType()))
])

movies_parsed = bronze_movies.withColumn(
    "data", from_json(col("raw_json"), movie_schema)
).select("data.*")

movies_parsed.createOrReplaceTempView("movies_parsed")
movies_parsed.show(5, truncate=True)



In [0]:
movies_exploded = movies_parsed.withColumn("genre_id", explode(col("genre_ids")))

movies_with_genre_names = movies_exploded.join(
    bronze_genres,
    movies_exploded.genre_id == bronze_genres.id,
    "left"
).select(
    movies_exploded["id"].alias("movie_id"),
    movies_exploded["title"],
    movies_exploded["overview"],
     movies_exploded["original_language"],
    movies_exploded["release_date"],
    movies_exploded["vote_average"],
      movies_exploded["vote_count"],
    movies_exploded["popularity"],
    bronze_genres["name"].alias("genre_name")
)




In [0]:
movies_with_language = movies_with_genre_names.join(
    spark.table("bronze_languages").select(
        col("iso_639_1").alias("original_language"),
        col("english_name").alias("language_name")
    ),
    on="original_language",
    how="left"
)


movies_with_language.write.format("delta").mode("overwrite").saveAsTable("moviebuff.default.silver_movie_genres")

In [0]:
%sql
select * FROM moviebuff.default.silver_movie_genres 

In [0]:
from pyspark.sql.functions import collect_list

dim_movies = movies_with_language.groupBy(
    "movie_id", "title","original_language", "overview", "release_date", "vote_average","vote_count", "popularity","language_name"
).agg(
    collect_list("genre_name").alias("genres")
)

dim_movies.write.format("delta").mode("overwrite").saveAsTable("moviebuff.default.silver_dim_movies")



In [0]:
provider_schema = StructType([
    StructField("link", StringType()),
    StructField("flatrate", ArrayType(StructType([
        StructField("provider_name", StringType())
    ])))
])

providers_parsed = bronze_providers.withColumn(
    "data", from_json(col("raw_json"), provider_schema)
).select(
    col("movie_id"),
    col("country"),
    col("data.link").alias("watch_link"),
    col("data.flatrate").alias("flatrate_list")
)

providers_exploded = providers_parsed.withColumn(
    "provider", explode(col("flatrate_list"))
).select(
    "movie_id",
    "country",
    "watch_link",
    col("provider.provider_name").alias("platform")
)





In [0]:
from pyspark.sql.functions import regexp_replace, trim

providers_normalized = providers_exploded.withColumn(
    "platform_clean",
    trim(
        regexp_replace(
            regexp_replace(
                regexp_replace(
                    regexp_replace(
                        regexp_replace(
                            regexp_replace(col("platform"), r"\s+with Ads$", ""),
                            r"\s+Amazon Channel$", ""
                        ),
                        r"\s+Apple TV Channel$", ""
                    ),
                    r"\s+Kids$", ""
                ),
                r"^Paramount\+$", "Paramount Plus"   # normalize + to "Plus"
            ),
            r"^Paramount Plus (Basic|Premium)$", "Paramount Plus"  # collapse tiers
        )
    )
)




In [0]:
providers_normalized.select('platform','platform_clean').distinct().display()

In [0]:
providers_normalized.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("moviebuff.default.silver_watch_providers")


In [0]:
## check movies and the watch providers
 
%sql

SELECT m.title, p.country, p.platform_clean
FROM silver_dim_movies m
JOIN silver_watch_providers p ON m.movie_id = p.movie_id
WHERE p.country = 'IN'
group by m.title, p.country, p.platform_clean
LIMIT 10;

In [0]:
%sql
SELECT DISTINCT platform, platform_clean FROM moviebuff.default.silver_watch_providers ORDER BY platform_clean;

In [0]:
%sql
SELECT DISTINCT m.title, g.genre_name, p.country, p.platform_clean
FROM silver_dim_movies m
JOIN silver_movie_genres g ON m.movie_id = g.movie_id
JOIN silver_watch_providers p ON m.movie_id = p.movie_id
WHERE g.genre_name = 'Action' AND p.country = 'IN'
LIMIT 10;